# LangChain RAG pipeline (modular)

Hadith RAG over `data/dorar_hadith_full_batch_2.csv`. Each section below is one pipeline stage; change **`CONFIG`** in the next cell to swap models, chunking, or retrieval without touching the rest.

**Run order:** Config → Imports → Load → Split → Embeddings → Vector store → Retriever → LLM → Prompt → Chain → Query.

**Requirements:** `pip install -r requirements.txt` (optional: copy `.env.example` to `.env` for API keys).

**Changes from base:**
- `load_hadith_documents` deduplicates hadiths per sharh group on `(hadith, rawy)` so duplicate CSV rows do not inflate metadata lists.
- `retrieve_filtered` uses **exact match** for controlled-vocab columns (`rawy`, `hokm`, `mohadth`, `source`) and **substring/contains match** for free-text columns (`categories`, …). `k_fetch` raised to `k_target × 10` so rare filters still return enough results.
- `EXACT_MATCH_COLUMNS` set makes the match-mode configurable without touching the filter loop.
- BLEU & ROUGE evaluation uses **`best_match` strategy**: ground-truth `Hadith_Matn` is compared against every retrieved hadith individually and the best score is reported — correct when retrieval returns multiple hadiths per chunk.
- **Tashkeel is stripped from both sides** before BLEU/ROUGE comparison via `normalize_arabic` (also normalizes أإآ→ا, ة→ه, ى→ي). Retrieved hadiths are pre-normalized at collection time so diacritical differences between the scraped index and the eval CSV do not inflate miss-rates.
- `strategy` config key in `EVAL_CONFIG["bleu_rouge"]` is now respected: `"best_match"` computes only best-match scores, `"concatenate"` computes only concat scores, anything else computes both.

In [1]:
# import re
# import pandas as pd
# TASHKEEL = re.compile(r'[\u0610-\u061A\u064B-\u065F\u06D6-\u06ED]')
#
# def normalize(text):
#     if pd.isna(text):
#         return text
#     text = re.sub(TASHKEEL, '', text)
#     return text
#
# df = pd.read_csv("data/dorar_hadith_full_batch_2.csv")
# df['sharh'] = df['sharh'].apply(normalize)
# df.to_csv("data/dorar_hadith_without_tashkeel.csv")

# Data

In [2]:
!python -m pip install -r /kaggle/input/datasets/aliabdelmenam/gg-data/requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 32.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 65.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 88.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 67.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━

In [3]:
import gdown

url = 'https://drive.google.com/uc?id=1QTOg3OeNSsJ0jh6llOw4BjeedyQZ2v-p'
gdown.download(url, quiet=False, use_cookies=False)

Downloading...
From (original): https://drive.google.com/uc?id=1QTOg3OeNSsJ0jh6llOw4BjeedyQZ2v-p
From (redirected): https://drive.google.com/uc?id=1QTOg3OeNSsJ0jh6llOw4BjeedyQZ2v-p&confirm=t&uuid=4f0a7445-60c1-4085-bb86-b949de7cdd3f
To: /kaggle/working/my_scraped_data.csv
100%|██████████| 847M/847M [00:05<00:00, 153MB/s]  


'my_scraped_data.csv'

In [4]:
import pandas as pd

gg = pd.read_csv("/kaggle/working/my_scraped_data.csv")
gg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 226389 entries, 0 to 226388
Data columns (total 11 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   page_id     226389 non-null  int64 
 1   url         226389 non-null  object
 2   categories  226373 non-null  object
 3   sharh       222599 non-null  object
 4   hadith      226389 non-null  object
 5   rawy        218543 non-null  object
 6   mohadth     226389 non-null  object
 7   source      226389 non-null  object
 8   page        226389 non-null  object
 9   hokm        226389 non-null  object
 10  takhrij     213849 non-null  object
dtypes: int64(1), object(10)
memory usage: 19.0+ MB


In [5]:
df = gg.copy(deep=True)

In [6]:
# Only remove brackets if the string starts with '[' AND ends with ']'
mask = df['hokm'].str.startswith('[', na=False) & df['hokm'].str.endswith(']', na=False)
df.loc[mask, 'hokm'] = df.loc[mask, 'hokm'].str[1:-1]

mask = df['rawy'].str.startswith('[', na=False) & df['rawy'].str.endswith(']', na=False)
df.loc[mask, 'rawy'] = df.loc[mask, 'rawy'].str[1:-1]

mask = df['mohadth'].str.startswith('[', na=False) & df['mohadth'].str.endswith(']', na=False)
df.loc[mask, 'mohadth'] = df.loc[mask, 'mohadth'].str[1:-1]

display(df['hokm'].value_counts().head(60))

hokm
صحيح                                                                       112359
إسناده صحيح                                                                 18841
حسن                                                                          9394
أخرجه في صحيحه                                                               7238
إسناده حسن                                                                   6049
إسناده صحيح على شرط الشيخين                                                  4491
حسن صحيح                                                                     3832
إسناده صحيح على شرط مسلم                                                     3442
صحيح لغيره                                                                   2732
ثابت                                                                         2572
إسناده جيد                                                                   2053
سكت عنه [وقد قال في رسالته لأهل مكة كل ما سكت عنه فهو صالح]                  1441
رجاله ثقات 

In [7]:
df.drop(['page_id','url','sharh'],axis=1).duplicated().sum()

np.int64(241)

# Config

In [8]:
from pathlib import Path

CONFIG = {
  "paths": {
    "data_csv": Path("/kaggle/working/my_scraped_data.csv"),
    "chroma_dir": Path("chroma_db"),
    "collection_name": "hadith_rag",
  },
  "data": {
    "max_rows": None,
    "text_columns": ["sharh"],
    "metadata_columns":
        ["page_id", "url", "categories", "hadith",
         "rawy", "mohadth",
         "source", "hokm", "page", "takhrij"],
  },
  "chunking": {
    "chunk_size": 350,
    "chunk_overlap": 35,
    # "separators": ["."],
  },
  "embeddings": {
    "provider": "huggingface",
    "model_name": "Omartificial-Intelligence-Space/Arabic-Triplet-Matryoshka-V2",
    "openai_model": "text-embedding-3-small",
  },
  "vector_store": {
    "persist": True,
    "reset_on_build": True,
  },
  "retriever": {
    "search_type": "similarity",
    "k": 8,
    "fetch_k": 25,
    "lambda_mult": 0.5,
  },
  "llm": {
    "provider": "huggingface_local",
    "openai_model": "gpt-4o-mini",
    "ollama_model": "llama3.2",
    "groq_model": "llama-3.3-70b-versatile",
    "huggingface_model": "silma-ai/SILMA-Kashif-2B-Instruct-v1.0", # Fixed Typo here!
    "temperature": 0.1,
  },
  "prompt": {
    "language": "ar",
    "system_role": (
      "أنت عالم دين إسلامي متخصص، مهمتك الإجابة عن أسئلة المستخدمين بناءً فقط على النص المُسند إليك. "
      "امتنع تمامًا عن تأليف أي أقوال دينية أو اجتهادات شخصية غير واردة في النص المقدّم. "
      "عند الاستشهاد بالأحاديث، الزم بتخريجها مع ذكر المصدر (الكتاب، رقم الحديث)، ودرجة صحتها (صحيح، حسن، ضعيف، موضوع) بحسب ما ورد في المصادر المعتمدة. "
      "إذا لم تجد إجابة وافية في السياق المتاح، أو كان السؤال خارج نطاق النص، فلا تفتِ من عندك، بل وجّه المستخدم إلى ضرورة استشارة عالم أو مؤسسة دينية موثوقة للحصول على فتوى دقيقة ومناسبة. "
      "التزم بالأدب الجم، والوضوح، والاختصار، مع الحرص على تعزيز الطمأنينة النفسية للمستخدم."
    ),
  },
}

PROJECT_ROOT = Path(".").resolve()

# Safe absolute path assignment
if not CONFIG["paths"]["data_csv"].is_absolute():
    CONFIG["paths"]["data_csv"] = PROJECT_ROOT / CONFIG["paths"]["data_csv"]

if not CONFIG["paths"]["chroma_dir"].is_absolute():
    CONFIG["paths"]["chroma_dir"] = PROJECT_ROOT / CONFIG["paths"]["chroma_dir"]

In [9]:
import os
import shutil
from typing import Any

import pandas as pd
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re
load_dotenv()

def cfg(*keys: str) -> Any:
    """Read nested CONFIG values, e.g. cfg('retriever', 'k')."""
    node = CONFIG
    for key in keys:
        node = node[key]
    return node

In [10]:
ARABIC_TASHKEEL = re.compile(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06ED]")
NON_WORD = re.compile(r"[^\w\s\u0600-\u06FF]+", re.UNICODE)


In [11]:

def normalize_arabic(text: str) -> str:
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return ""
    text = str(text)
    text = ARABIC_TASHKEEL.sub("", text)
    text = text.replace("\ufeff", "")
    text = re.sub(r"[إأآٱ]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ة", "ه", text)
    text = NON_WORD.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


In [ ]:
import ast
from typing import Union
import pandas as pd
from pathlib import Path
from langchain_core.documents import Document

# ── Constants ────────────────────────────────────────────────
HADITH_LEVEL_LIST_COLUMNS = ["hadith", "rawy", "mohadth", "source", "hokm", "takhrij",
                              "page_id", "url", "categories", "page"]
EMBED_COLUMN = "sharh"
SCALAR_META_COLUMNS = []

# ── Helpers ─────────────────────────────────────────────────
def _parse_to_list(val) -> list[str]:
    if isinstance(val, list): return [str(x) for x in val]
    if not isinstance(val, str): return [str(val)] if pd.notna(val) else [""]
    s = val.strip()
    if s.startswith("["):
        try:
            res = ast.literal_eval(s)
            return [str(x) for x in res] if isinstance(res, list) else [str(res)]
        except: pass
    return [s]

def load_hadith_documents(csv_path=None, max_rows: int | None = None) -> list[Document]:
    csv_path = Path(csv_path or cfg("paths", "data_csv"))

    df = pd.read_csv(csv_path)

    if max_rows:
        df = df.head(max_rows)

    df = df[df[EMBED_COLUMN].notna()]

    docs = []

    for sharh_text, group in df.groupby(EMBED_COLUMN, sort=False):

        group_records = []

        for _, row in group.iterrows():

            row_data = {
                col: _parse_to_list(row.get(col, ""))
                for col in HADITH_LEVEL_LIST_COLUMNS
            }

            n = len(row_data["hadith"])

            for i in range(n):
                record = {
                    col: (
                        row_data[col][i]
                        if i < len(row_data[col])
                        else ""
                    )
                    for col in HADITH_LEVEL_LIST_COLUMNS
                }

                group_records.append(record)

        metadata = {}

        for col in HADITH_LEVEL_LIST_COLUMNS:
            metadata[col] = [rec[col] for rec in group_records]

        docs.append(
            Document(
                page_content=str(sharh_text).strip(),
                metadata=metadata,
            )
        )

    print(f"Loaded {len(docs)} documents with proper metadata alignment.")
    return docs


from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# In-memory parent store — full parent docs indexed by parent_id
PARENT_STORE: list[Document] = []
from transformers import AutoTokenizer
from langchain_core.documents import Document

PARENT_STORE: list[Document] = []

def build_child_chunks(
    raw_docs: list[Document],
    chunk_size: int = 350,
    chunk_overlap: int = 32,
    model_name: str = "Omartificial-Intelligence-Space/Arabic-Triplet-Matryoshka-V2",
) -> list[Document]:
    global PARENT_STORE
    PARENT_STORE = []

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def split_by_tokens(text: str) -> list[str]:
        tokens = tokenizer.encode(text, add_special_tokens=False)
        chunks = []
        step = chunk_size - chunk_overlap
        for i in range(0, len(tokens), step):
            chunk_tokens = tokens[i : i + chunk_size]
            chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
            if chunk_text.strip():
                chunks.append(chunk_text.strip())
        return chunks

    child_docs: list[Document] = []

    for parent_id, parent_doc in enumerate(raw_docs):
        PARENT_STORE.append(parent_doc)

        chunks = split_by_tokens(parent_doc.page_content)

        for chunk_idx, chunk_text in enumerate(chunks):
            child_meta = {
                "parent_id":   parent_id,
                "chunk_index": chunk_idx,
                "rawy":        parent_doc.metadata.get("rawy", []),
                "hokm":        parent_doc.metadata.get("hokm", []),
                "mohadth":     parent_doc.metadata.get("mohadth", []),
                "source":      parent_doc.metadata.get("source", []),
                "categories":  parent_doc.metadata.get("categories", []),
            }
            child_docs.append(Document(page_content=chunk_text, metadata=child_meta))

    print(f"Parents: {len(PARENT_STORE)} | Child chunks: {len(child_docs)}")
    return child_docs

In [13]:
# print(len(raw_documents[0].metadata['hadith']))
# print(len(raw_documents[0].metadata['url']))

In [14]:
# ── Pre-index filter ─────────────────────────────────────────────────────────
ACTIVE_FILTERS: dict = {
    # "rawy": ["البخاري", "مسلم"],
    # "hokm": "صحيح",
}

# Fix: Load the documents first so 'raw_documents' is defined
raw_documents = load_hadith_documents(
    cfg("paths", "data_csv"),
    max_rows=cfg("data", "max_rows")
)

# Note: This code expects 'filter_documents' to be defined.
# If it was removed, we use the documents directly or define a simple filter helper.
def filter_documents(docs, **filters):
    if not filters: return docs
    norm = {k: ([v.lower()] if isinstance(v, str) else [x.lower() for x in v]) for k, v in filters.items()}
    return [d for d in docs if all(any(f in str(d.metadata.get(k, '')).lower() for f in allowed) for k, allowed in norm.items())]

# documents_to_index = (
#     filter_documents(raw_documents, **ACTIVE_FILTERS)
#     if ACTIVE_FILTERS
#     else raw_documents
# )
filtered_parents = (
    filter_documents(raw_documents, **ACTIVE_FILTERS)
    if ACTIVE_FILTERS
    else raw_documents
)
documents_to_index = build_child_chunks(
    filtered_parents,
    chunk_size=cfg("chunking", "chunk_size"),
    chunk_overlap=cfg("chunking", "chunk_overlap"),
)

# print(f"\nDocuments to index: {len(documents_to_index)}")
# print(documents_to_index[0].page_content[:300] if documents_to_index else "— empty —")
print(f"Child chunks to index: {len(documents_to_index)}")
# print(documents_to_index[0].page_content[:200] if documents_to_index else "— empty —")

Loaded 14691 documents with proper metadata alignment.


config.json:   0%|          | 0.00/637 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1183 > 512). Running this sequence through the model will result in indexing errors


Parents: 14691 | Child chunks: 22643
Child chunks to index: 22643


In [15]:
len(documents_to_index)

22643

In [16]:
len(raw_documents)

14691

In [17]:
raw_documents[0].page_content

'أرسل الله تعالى نبيه محمدا صلى الله عليه وسلم رحمة للعالمين، وجعل في امتثال أمره واجتناب نهيه النجاة في الدنيا والآخرة، فكانت طريقته التيسير على الناس في عباداتهم وحياتهم. وفي هذا الحديث يروي أبو هريرة رضي الله عنه أن رسول الله صلى الله عليه وسلم رأى رجلا ماشيا على قدميه، ويسوق أمامه بدنة قد أهداها إلى البيت الحرام يتقرب بها إلى الله تعالى، والبدنة: تكون من الإبل خاصة، وقيل: البدن تطلق على الإبل والبقر. فأمره النبي صلى الله عليه وسلم بركوبها؛ ليستريح من تعبه الذي حصل له من مشقة المشي، فأخبره الرجل أنها بدنة مهداة إلى الكعبة، فكيف يركبها؟ فقال له النبي صلى الله عليه وسلم في المرة الثانية أو الثالثة: «اركبها، ويلك!» وأصل الويل: العذاب الشديد، وهو غير مقصود هنا، وإنما أراد النبي صلى الله عليه وسلم أن يغلظ له في القول ليركبها. وفي الحديث: مشروعية ركوب الهدي. وفيه: الندب إلى المبادرة إلى امتثال أمر الله ورسوله، وزجر من لم يبادر إلى ذلك، وتوبيخه.'

In [18]:
len(raw_documents[0].metadata['hadith'])

31

In [19]:
print(len(documents_to_index[0].metadata['rawy']))
print(len(documents_to_index[0].metadata['source']))
print(len(documents_to_index[0].metadata['hokm']))

31
31
31


In [20]:
documents_to_index[0].metadata.keys()

dict_keys(['parent_id', 'chunk_index', 'rawy', 'hokm', 'mohadth', 'source', 'categories'])

In [21]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.device_count())

2.10.0+cu128
12.8
True
2


In [22]:
!pip install langchain_huggingface

In [23]:
def build_embeddings():
    provider = cfg("embeddings", "provider")
    if provider == "openai":
        from langchain_openai import OpenAIEmbeddings
        return OpenAIEmbeddings(model=cfg("embeddings", "openai_model"))
    if provider == "huggingface":
        from langchain_huggingface import HuggingFaceEmbeddings
        import torch

        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Using device: {device}")

        return HuggingFaceEmbeddings(
            model_name=cfg("embeddings", "model_name"),
            model_kwargs={"device": device},
            encode_kwargs={"batch_size": 256},
        )
    raise ValueError(f"Unknown embeddings provider: {provider}")


embeddings = build_embeddings()
print(f"Embeddings ready: {cfg('embeddings', 'provider')} / {cfg('embeddings', 'model_name') if cfg('embeddings', 'provider') == 'huggingface' else cfg('embeddings', 'openai_model')}")

Using device: cuda


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/541M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Embeddings ready: huggingface / Omartificial-Intelligence-Space/Arabic-Triplet-Matryoshka-V2


In [24]:
!pip install langchain_chroma

In [25]:
from langchain_chroma import Chroma
import shutil

chroma_dir = cfg("paths", "chroma_dir")
collection = cfg("paths", "collection_name")

if cfg("vector_store", "reset_on_build") and chroma_dir.exists():
    shutil.rmtree(chroma_dir)
    print(f"Removed {chroma_dir}")

# Updated to use Cosine Similarity
vectorstore = Chroma(
    collection_name=collection,
    embedding_function=embeddings,
    persist_directory=str(chroma_dir) if cfg("vector_store", "persist") else None,
    collection_metadata={"hnsw:space": "cosine"}
)

def _collection_has_vectors(vs: Chroma) -> bool:
    data = vs.get(limit=1)
    return bool(data.get("ids"))

if not _collection_has_vectors(vectorstore):
    from tqdm import tqdm

    if 'documents_to_index' not in globals():
        print("Variable 'documents_to_index' not found. Defaulting to 'raw_documents'.")
        documents_to_index = raw_documents

    batch_size = 512
    for i in tqdm(range(0, len(documents_to_index), batch_size)):
        vectorstore.add_documents(documents_to_index[i : i + batch_size])

    print(f"Indexed {len(documents_to_index)} documents into '{collection}' with Cosine Similarity")
else:
    n = len(vectorstore.get().get("ids", []))
    print(f"Using existing index: {n} vectors in '{collection}'")

vectorstore

100%|██████████| 45/45 [08:32<00:00, 11.40s/it]

Indexed 22643 documents into 'hadith_rag' with Cosine Similarity


## Retriever — per-hadith aligned metadata filtering

**Fix #1:** `retrieve_filtered` now trims metadata down to only the rows that match
the active filter.  If `rawy=["البخاري"]` and a chunk has 10 hadiths of which 3
belong to البخاري, only those 3 hadiths are returned — along with their paired
`mohadth`, `source`, `hokm`, and `takhrij` at the same list index.

In [26]:
from dataclasses import dataclass, field
import ast
import pandas as pd


@dataclass
class RetrievedResult:
    """
    One entry returned by retrieve_filtered().
    """
    sharh: str
    hadiths: list[str]
    list_meta: dict = field(default_factory=dict)
    scalar_meta: dict = field(default_factory=dict)
    score: float | None = None

    def __repr__(self):
        preview = self.sharh[:80].replace("\n", " ")
        return (
            f"RetrievedResult(\n"
            f"  hadiths_count={len(self.hadiths)},\n"
            f"  rawy={self.list_meta.get('rawy', 'N/A')!r},\n"
            f"  hokm={self.list_meta.get('hokm', 'N/A')!r},\n"
            f"  categories={self.list_meta.get('categories', 'N/A')!r},\n"
            f"  url={self.list_meta.get('url', 'N/A')!r},\n"
            f"  page={self.list_meta.get('page', 'N/A')!r},\n"
            f"  sharh_preview={preview!r}...\n"
            f")"
        )

# All hadith-level columns stored as parallel lists in each Document's metadata
HADITH_LEVEL_LIST_COLUMNS = ["hadith", "rawy", "mohadth", "source", "hokm", "takhrij",
                              "page_id", "url", "categories", "page"]

SHARH_LEVEL_SCALAR_COLUMNS = []

# Columns where filtering should be an EXACT match (controlled vocabulary).
# Everything else uses substring / contains matching (free-text fields).
EXACT_MATCH_COLUMNS = {"rawy", "hokm", "mohadth", "source"}

def _safe_parse_list(raw) -> list[str]:
    """Safely coerce a Chroma metadata value back to a Python list of strings.
    Chroma stores lists as their repr strings, so we always need ast.literal_eval."""
    if isinstance(raw, list):
        return [str(v) if (v and pd.notna(v)) else "" for v in raw]
    if isinstance(raw, str) and raw.startswith("["):
        try:
            parsed = ast.literal_eval(raw)
            return [str(v) if (v and pd.notna(v)) else "" for v in parsed]
        except Exception:
            pass
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return [""]
    return [str(raw)]

def _build_row_records(meta: dict) -> list[dict]:
    parsed: dict[str, list[str]] = {
        col: _safe_parse_list(meta.get(col, []))
        for col in HADITH_LEVEL_LIST_COLUMNS
    }
    n_rows = len(parsed.get("hadith", []))
    if n_rows == 0: return []

    records = []
    for i in range(n_rows):
        row = {}
        for col in HADITH_LEVEL_LIST_COLUMNS:
            vals = parsed[col]
            row[col] = vals[i] if i < len(vals) else ""
        records.append(row)
    return records

def _filter_row_records(records: list[dict], norm_filters: dict[str, list[str]]) -> list[dict]:
    """
    AND across columns, OR within a column's allowed list.
    - Controlled-vocab columns (rawy, hokm, …): exact equality match.
    - Free-text columns (categories, …): substring / contains match.
    """
    if not norm_filters: return records
    kept = []
    for rec in records:
        passes = True
        for col, allowed in norm_filters.items():
            val = str(rec.get(col, "")).lower()
            if col in EXACT_MATCH_COLUMNS:
                # Exact match: the cell value must equal one of the allowed values
                if not any(a == val for a in allowed):
                    passes = False
                    break
            else:
                # Substring match: the cell value must contain one of the allowed values
                if not any(a in val for a in allowed):
                    passes = False
                    break
        if passes: kept.append(rec)
    return kept

def _records_to_list_meta(records: list[dict]) -> dict[str, list[str]]:
    if not records: return {col: [] for col in HADITH_LEVEL_LIST_COLUMNS}
    result: dict[str, list[str]] = {col: [] for col in HADITH_LEVEL_LIST_COLUMNS}
    for rec in records:
        for col in HADITH_LEVEL_LIST_COLUMNS:
            result[col].append(rec.get(col, ""))
    return result

# def retrieve_filtered(query: str, **filters) -> list[RetrievedResult]:
#     """
#     Retrieve up to cfg('retriever', 'k') results that match all supplied filters.
#     Filters are AND-combined across columns; within a column a list of values is OR-combined.
#     k_fetch is set to k_target * 10 so that post-filter we still get k_target hits
#     even when the filter is selective (e.g. a rare rawy value).
#     """
#     k_target = cfg("retriever", "k")
#     # Fetch 10× more than needed; rare filters may discard most candidates
#     k_fetch = k_target * 10
#     norm_filters = {
#         col: ([v.lower() for v in val] if isinstance(val, list) else [val.lower()])
#         for col, val in filters.items()
#     }
#     raw_docs = vectorstore.similarity_search(query, k=k_fetch)
#     results: list[RetrievedResult] = []
#     for doc in raw_docs:
#         records = _build_row_records(doc.metadata)
#         if not records: continue
#         kept_records = _filter_row_records(records, norm_filters)
#         if not kept_records: continue
#         filtered_list_meta = _records_to_list_meta(kept_records)
#         hadiths = filtered_list_meta.pop("hadith", [])
#         if not hadiths: continue
#         results.append(RetrievedResult(
#             sharh=doc.page_content,
#             hadiths=hadiths,
#             list_meta=filtered_list_meta,
#             scalar_meta={k: v for k, v in doc.metadata.items() if k in SHARH_LEVEL_SCALAR_COLUMNS}
#         ))
#         if len(results) >= k_target: break
#     return results
def retrieve_filtered(query: str, **filters) -> list[RetrievedResult]:
    k_target = cfg("retriever", "k")
    k_fetch  = k_target * 10

    norm_filters = {
        col: ([v.lower() for v in val] if isinstance(val, list) else [val.lower()])
        for col, val in filters.items()
    }

    raw_docs = vectorstore.similarity_search(query, k=k_fetch)

    results: list[RetrievedResult] = []
    seen_parents: set[int] = set()

    for doc in raw_docs:
        parent_id = doc.metadata.get("parent_id")
        if parent_id is None:
            continue

        # Stage 1: chunk-level filter using copied metadata
        chunk_passes = True
        for col, allowed in norm_filters.items():
            col_vals = doc.metadata.get(col, [])
            if isinstance(col_vals, str):
                try:
                    col_vals = ast.literal_eval(col_vals)
                except Exception:
                    col_vals = [col_vals]
            col_vals_lower = [str(v).lower() for v in col_vals]

            if col in EXACT_MATCH_COLUMNS:
                if not any(a in col_vals_lower for a in allowed):
                    chunk_passes = False
                    break
            else:
                if not any(any(a in v for a in allowed) for v in col_vals_lower):
                    chunk_passes = False
                    break

        if not chunk_passes:
            continue

        # Stage 2: deduplicate by parent_id
        if parent_id in seen_parents:
            continue
        seen_parents.add(parent_id)

        # Stage 3: fetch full parent from memory
        parent_doc = PARENT_STORE[parent_id]

        # Stage 4: filter parent's hadiths list
        records = _build_row_records(parent_doc.metadata)
        kept_records = _filter_row_records(records, norm_filters)
        if not kept_records:
            continue

        # # Stage 5: pick longest hadith (after filter)
        # kept_records = [max(
        #     kept_records,
        #     key=lambda r: len(normalize_arabic(r.get("hadith", "")).split())
        # )]

        filtered_list_meta = _records_to_list_meta(kept_records)
        hadiths = filtered_list_meta.pop("hadith", [])
        if not hadiths:
            continue

        results.append(RetrievedResult(
            sharh=parent_doc.page_content,
            hadiths=hadiths,
            list_meta=filtered_list_meta,
            scalar_meta={},
        ))

        if len(results) >= k_target:
            break

    return results
def print_results(results: list[RetrievedResult], max_hadith_preview: int = 120) -> None:
    for i, r in enumerate(results, 1):
        print(f"{' ─'*60}")
        def get_meta(key):
            val = r.list_meta.get(key, [])
            return [v if v.strip() else "Unknown" for v in val]

        print(f"[{i}] rawy={get_meta('rawy')}")
        print(f"    hokm={get_meta('hokm')}")
        print(f"    source={get_meta('source')}")
        print(f"    categories={r.list_meta.get('categories', 'N/A')}")
        print(f"    url={r.list_meta.get('url', 'N/A')}")
        print(f"    page={r.list_meta.get('page', 'N/A')}")
        print(f"    sharh[:150]: {r.sharh[:150].replace(chr(10),' ')}…")
        print(f"    hadiths ({len(r.hadiths)}):")
        for j, h in enumerate(r.hadiths):
            rawy_list = r.list_meta.get('rawy', [])
            label = rawy_list[j] if (j < len(rawy_list) and rawy_list[j].strip()) else 'Unknown'
            print(f"      [{j+1}] rawy={label!r} \n {h[:max_hadith_preview]}")
    print(f"{' ─'*60}")

In [27]:
def top_w_hadiths(results: list[RetrievedResult], w: int) -> list[RetrievedResult]:
    """
    For each RetrievedResult chunk, keep only the top-w hadiths with the most words.
    
    Args:
        results: List of RetrievedResult objects from retrieve_filtered().
        w: Maximum number of hadiths to keep per chunk (ranked by word count, descending).
    
    Returns:
        A new list of RetrievedResult objects, each with at most w hadiths.
        Chunks that end up with zero hadiths are dropped.
    
    Example:
        k=5 chunks × 20 hadiths each → 100 total
        top_w_hadiths(results, w=3) → at most 5×3 = 15 hadiths
    """
    if w <= 0:
        raise ValueError(f"w must be a positive integer, got {w}")

    trimmed_results: list[RetrievedResult] = []

    for r in results:
        if not r.hadiths:
            continue

        # Pair each hadith with its word count and original index
        indexed = [(i, h, len(normalize_arabic(h).split())) for i, h in enumerate(r.hadiths)]

        # Sort by word count descending, keep top-w
        top_indices = sorted(indexed, key=lambda x: x[2], reverse=True)[:w]

        # Re-sort by original index to preserve relative order
        top_indices.sort(key=lambda x: x[0])

        selected_indices = [idx for idx, _, _ in top_indices]

        # Rebuild hadiths list
        new_hadiths = [r.hadiths[i] for i in selected_indices]

        # Rebuild list_meta: slice each parallel list to the selected indices
        new_list_meta: dict[str, list] = {}
        for col, values in r.list_meta.items():
            new_list_meta[col] = [values[i] for i in selected_indices if i < len(values)]

        trimmed_results.append(RetrievedResult(
            sharh=r.sharh,
            hadiths=new_hadiths,
            list_meta=new_list_meta,
            scalar_meta=r.scalar_meta,
            score=r.score,
        ))

    return trimmed_results


In [28]:
def build_retriever():
    search_type = cfg("retriever", "search_type")
    k = cfg("retriever", "k")

    if search_type == "similarity":
        return vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": k})
    if search_type == "mmr":
        return vectorstore.as_retriever(
            search_type="mmr",
            search_kwargs={
                "k": k,
                "fetch_k": cfg("retriever", "fetch_k"),
                "lambda_mult": cfg("retriever", "lambda_mult"),
            },
        )
    raise ValueError(f"Unknown search_type: {search_type}")


retriever = build_retriever()
print(f"Retriever: {cfg('retriever', 'search_type')}, k={cfg('retriever', 'k')}")

Retriever: similarity, k=8


In [29]:
!pip install langchain-huggingface transformers torch accelerate
!pip install langchain-openai langchain-community langchain-groq

In [30]:
import os
import torch
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace, HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

def build_llm():
    """
    Build and return an LLM instance based on configuration.
    Supports: openai, ollama, groq, huggingface (endpoint & local)
    """
    provider = cfg("llm", "provider")
    temperature = cfg("llm", "temperature")
    print(provider)
    try:
        # --- OpenAI ---
        if provider == "openai":
            from langchain_openai import ChatOpenAI
            return ChatOpenAI(
                model=cfg("llm", "openai_model"), 
                temperature=temperature
            )
        
        # --- Ollama (local) ---
        if provider == "ollama":
            from langchain_community.chat_models import ChatOllama
            return ChatOllama(
                model=cfg("llm", "ollama_model"), 
                temperature=temperature
            )
        
        # --- Groq ---
        if provider == "groq":
            from langchain_groq import ChatGroq
            api_key = _get_api_key('GROQ_API_KEY')
            return ChatGroq(
                model=cfg("llm", "groq_model"), 
                temperature=temperature,
                groq_api_key=api_key,
            )
        
        # --- HuggingFace (via Inference API) ---
        if provider == "huggingface":
            api_key = _get_api_key('HF_TOKEN')
            repo_id = cfg("llm", "huggingface_model")
            
            llm_endpoint = HuggingFaceEndpoint(
                repo_id=repo_id,
                temperature=temperature,
                huggingfacehub_api_token=api_key,
                task="text-generation",
                max_new_tokens=CONFIG["llm"].get("max_new_tokens", 512),
            )
            return ChatHuggingFace(llm=llm_endpoint)
        
        # --- HuggingFace (Local pipeline) ---
        if provider == "huggingface_local":
            model_name = cfg("llm", "huggingface_model")
            print(f"Model name: {model_name}")
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.float16,
                device_map="auto",
                low_cpu_mem_usage=True,
            )
            
            pipe = pipeline(
                "text-generation",
                model=model,
                tokenizer=tokenizer,
                temperature=temperature,
            )
            
            llm_pipeline = HuggingFacePipeline(pipeline=pipe)
            return ChatHuggingFace(llm=llm_pipeline)
            
    except Exception as e:  
        raise e 

def _get_api_key(key_name):
    """Helper to get API key from Kaggle Secrets or environment variables"""
    # 1. Try Kaggle Secrets
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(key_name)
    except Exception:
        pass
        
    # 2. Fallback to OS Environment Variables
    return os.environ.get(key_name)

# --- Load LLM ---
try:
    llm = build_llm()
    if llm is not None:
        print(f"✅ LLM Loaded: {cfg('llm', 'provider')}")
    else:
        print("❌ Error: build_llm returned None unexpectedly.")
except Exception as e:
    print(f"❌ Error initializing LLM: {e}")
    llm = None

huggingface_local
Model name: silma-ai/SILMA-Kashif-2B-Instruct-v1.0


config.json:   0%|          | 0.00/974 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/34.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✅ LLM Loaded: huggingface_local


In [31]:
def format_docs(docs: list[Document]) -> str:
    """
    Format retrieved Documents for the LLM prompt, including hadiths in context.
    """
    blocks = []
    for i, doc in enumerate(docs, 1):
        meta = doc.metadata
        rawy   = meta.get("rawy", "")
        hokm   = meta.get("hokm", "")
        source = meta.get("source", "")
        page_id = meta.get("page_id", "")

        hadiths_raw = meta.get("hadith", [])
        if isinstance(hadiths_raw, str):
            try:
                hadiths_raw = ast.literal_eval(hadiths_raw)
            except Exception:
                hadiths_raw = [hadiths_raw]

        hadith_block = ""
        if hadiths_raw:
            hadith_lines = "\n".join(f"  - {h}" for h in hadiths_raw[:5])
            hadith_block = f"\nالأحاديث المرتبطة:\n{hadith_lines}"

        header = f"[{i}] page_id={page_id} | راوي={rawy} | حكم={hokm} | مصدر={source}"
        blocks.append(f"{header}\n{doc.page_content}{hadith_block}")

    return "\n\n---\n\n".join(blocks)


RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system", cfg("prompt", "system_role")),
    (
        "human",
        "السياق المسترجع:\n{context}\n\n"
        "السؤال: {question}\n\n"
        f"أجب باللغة: {cfg('prompt', 'language')}. اذكر page_id عند الاقتباس إن وُجد.",
    ),
])

prompt = RAG_PROMPT
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='أنت عالم دين إسلامي متخصص، مهمتك الإجابة عن أسئلة المستخدمين بناءً فقط على النص المُسند إليك. امتنع تمامًا عن تأليف أي أقوال دينية أو اجتهادات شخصية غير واردة في النص المقدّم. عند الاستشهاد بالأحاديث، الزم بتخريجها مع ذكر المصدر (الكتاب، رقم الحديث)، ودرجة صحتها (صحيح، حسن، ضعيف، موضوع) بحسب ما ورد في المصادر المعتمدة. إذا لم تجد إجابة وافية في السياق المتاح، أو كان السؤال خارج نطاق النص، فلا تفتِ من عندك، بل وجّه المستخدم إلى ضرورة استشارة عالم أو مؤسسة دينية موثوقة للحصول على فتوى دقيقة ومناسبة. التزم بالأدب الجم، والوضوح، والاختصار، مع الحرص على تعزيز الطمأنينة النفسية للمستخدم.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='السياق المسترجع:\n{co

# RAG

In [32]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.documents import Document

# Assuming your vector_store instance was already initialized as `vector_store`
# e.g., vector_store = Chroma(persist_directory=cfg("paths", "chroma_dir"), ...)

QUERY_FILTERS: dict = {}

def retrieve(question: str, k: int | None = None, **extra_filters) -> list[Document]:
    """
    Retrieve documents matching the user question from the vector store, 
    applying specified configuration settings and metadata filters.
    """
    # Fall back to your config if k is not explicitly provided
    search_k = k if k is not None else CONFIG["retriever"]["k"]
    search_type = CONFIG["retriever"]["search_type"]
    
    # Merge global filters with any query-time specific filters
    search_filters = {**QUERY_FILTERS, **extra_filters}
    
    # Configure the search arguments
    search_kwargs = {"k": search_k}
    if search_filters:
        search_kwargs["filter"] = search_filters
        
    if search_type == "mmr":
        search_kwargs["fetch_k"] = CONFIG["retriever"]["fetch_k"]
        search_kwargs["lambda_mult"] = CONFIG["retriever"]["lambda_mult"]

    # Initialize the base retriever dynamically based on configuration
    retriever = vectorstore.as_retriever(
        search_type=search_type,
        search_kwargs=search_kwargs
    )
    
    return retriever.invoke(question)


def format_docs(docs: list[Document]) -> str:
    """
    Formats the list of documents into a unified context string,
    including the page_id as requested by the system prompt.
    """
    formatted_chunks = []
    for doc in docs:
        page_id = doc.metadata.get("page_id", "غير محدد")
        chunk_text = f"--- [معرف الصفحة: {page_id}] ---\n{doc.page_content}"
        formatted_chunks.append(chunk_text)
        
    return "\n\n".join(formatted_chunks)


# --- Rebuilding your LCEL Chain safely ---

# Define templates exactly matching your LangChain structure
system_template = CONFIG["prompt"]["system_role"]
human_template = "السياق المسترجع:\n{context}\n\nالسؤال: {question}\n\nأجب باللغة: ar. اذكر page_id عند الاقتباس إن وُجد."

prompt_template = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(system_template),
    HumanMessagePromptTemplate.from_template(human_template)
])

# Ensure your llm instance is initialized (from ChatHuggingFace wrapper)
# rag_llm = build_llm() 

# Complete LCEL Pipeline Chain
rag_chain = (
    {
        "context": RunnableLambda(lambda inputs: retrieve(inputs["question"])) | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | prompt_template
    | llm  # Your ChatHuggingFace pipeline wrapper instance
    | StrOutputParser()
)

In [33]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document

QUERY_FILTERS: dict = {}

def retrieve(question: str, k: int | None = None, **extra_filters) -> list[Document]:
    """
    Retrieve documents for a question.
    """
    active_filters = {**QUERY_FILTERS, **extra_filters}

    # Fetch configuration default value if k is not passed explicitly
    target_k = k if k is not None else CONFIG["retriever"]["k"]

    if active_filters:
        # Assumes retrieve_filtered is defined elsewhere in your codebase
        results = retrieve_filtered(question, **active_filters)
        docs = []
        for r in results:
            # Reconstruct metadata keys based on your specific backend objects
            meta = {**r.scalar_meta, **r.list_meta, "hadith": r.hadiths}
            docs.append(Document(page_content=r.sharh, metadata=meta))
        
        if target_k:
            docs = docs[:target_k]
        return docs
    else:
        # Assumes the base retriever instance is initialized globally
        docs = retriever.invoke(question)
        if target_k:
            docs = docs[:target_k]
        return docs

# --- Format Documents Helper ---
def format_docs(docs: list[Document]) -> str:
    """Formats retrieved document segments into a unified context block."""
    formatted_chunks = []
    for doc in docs:
        # Accessing page_id safely since your metadata dictionary structure packs scalar_meta
        page_id = doc.metadata.get("page_id", "غير محدد")
        chunk_text = f"--- [معرف الصفحة: {page_id}] ---\n{doc.page_content}"
        formatted_chunks.append(chunk_text)
    return "\n\n".join(formatted_chunks)

# --- Complete LCEL RAG Pipeline ---
rag_chain = (
    {
        # CRITICAL FIX: Extract just the 'question' string key from the dictionary 
        # instead of passing the whole dict into your retrieve function.
        "context": RunnableLambda(lambda inputs: retrieve(inputs["question"])) | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt  # Your ChatPromptTemplate instance
    | llm     # Your ChatHuggingFace pipeline wrapper instance
    | StrOutputParser()
)

# Preview the chain structure
rag_chain;

In [34]:
import ast
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def ask(question: str, *, show_sources: bool = True, **filters) -> str:
    # 1. Merge global and local filters
    active_filters = {**QUERY_FILTERS, **filters}

    # 2. Print metadata/sources if requested
    if show_sources:
        print("--- Retrieved chunks ---")
        docs = retrieve(question, **filters)
        for doc in docs:
            meta = doc.metadata
            hadiths = meta.get("hadith", [])
            
            # Safely handle stringified lists from the vector database metadata
            if isinstance(hadiths, str):
                try:
                    hadiths = ast.literal_eval(hadiths)
                except Exception:
                    hadiths = [hadiths]  # Fallback if parsing fails
                    
            print(
                f"page_id={meta.get('page_id')} | "
                f"rawy={meta.get('rawy')} | "
                f"hokm={meta.get('hokm')} | "
                f"sharh[:100]={doc.page_content[:100]}..."
            )
            for h in hadiths[:2]:
                print(f"  hadith: {h[:120]}")
                
        if active_filters:
            print(f"Active filters: {active_filters}")
        print("--- Answer ---")

    # 3. Handle ad-hoc query filters by constructing a temporary dynamic chain
    if filters:
        temp_chain = (
            {
                # Fixed: Properly extract the "question" string from the incoming input dictionary
                "context": (lambda inputs: retrieve(inputs["question"], **filters)) | format_docs,
                "question": RunnablePassthrough()
            }
            | prompt
            | llm
            | StrOutputParser()
        )
        # Fixed: Pass a structured dictionary instead of a raw string
        return temp_chain.invoke({"question": question})

    # 4. Fallback to default pipeline if no query-time filters are applied
    # Fixed: Pass a structured dictionary here as well
    return rag_chain.invoke({"question": question})


# --- Run Test Query ---
QUESTION = "ما حكم سجود السهو إذا زاد الإمام في الصلاة؟"
answer = ask(QUESTION)
print(answer)

--- Retrieved chunks ---
page_id=None | rawy=['عبدالله بن مسعود', 'عبدالله بن مسعود', 'عبد الله', 'عبدالله بن مسعود', 'عبدالله بن مسعود', '', 'عبدالله بن مسعود'] | hokm=['إسناده صحيح على شرط مسلم', 'إسناده صحيح على شرط الشيخين', 'إسناده ثابت', '[صحيح]', 'صحيح', 'صحيح', '[صحيح]'] | sharh[:100]=إذا سها الإمام في الصلاة. ثم قال صلى الله عليه وسلم للناس : « وإذا شك أحدكم » فنسي في صلاته فلم يدر ...
page_id=None | rawy=['معاوية بن أبي سفيان', 'يوسف مولى عثمان', 'عبدالله بن جعفر', 'معاوية', 'معاوية بن أبي سفيان', 'معاوية بن أبي سفيان'] | hokm=['صحيح لغيره', 'صحيح لغيره', 'أخرجه في صحيحه', 'إسناده جيد', '[إسناده] حسن', '[إسناده] حسن'] | sharh[:100]=الصلاة عماد الدين ، وعلى المسلم أن يلتزم اتباع النبي صلى الله عليه وسلم في أدائها ، وقد علمها أصحابه...
page_id=None | rawy=['عبدالله بن مسعود'] | hokm=['[صحيح]'] | sharh[:100]=الصلاة عماد الدين ، وعلى العبد أن يلزم فيها الخشوع والتدبر ، وعدم الانشغال بأحوال الدنيا ، ولكنه قد ...
page_id=None | rawy=['المغيرة بن شعبة', 'المغيرة بن شعبة', 'المغيرة ب

TemplateError: System role not supported

# Retrieve

In [35]:
# Test standalone retrieval with filtering
query = "ايه هو سجود السهو؟"

# Use retrieve_filtered directly to get RetrievedResult objects for print_results
results = retrieve_filtered(query)


print(f"Found {len(results)} results for: '{query}' with source filter\n")
print_results(results,max_hadith_preview=-1)

Found 8 results for: 'ايه هو سجود السهو؟' with source filter

 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
[1] rawy=['Unknown', 'أبو سعيد الخدري', 'Unknown']
    hokm=['صحيح', 'ثابت', 'صحيح']
    source=['الفتاوى الكبرى', 'التمهيد', 'الفتاوى الكبرى']
    categories=['سهو - إذا لم يدر كم صلى ، سهو - السهو في الفرض والتطوع ، صلاة - الوسوسة في الصلاة ، إيمان - الجن والشياطين ، سهو - سجود السهو قبل التسليم', 'جن - صفة إبليس وجنوده ، سهو - إذا لم يدر كم صلى ، آداب عامة - الخطأ والنسيان ، إيمان - أعمال الجن والشياطين ، إيمان - الجن والشياطين', 'جن - صفة إبليس وجنوده ، سهو - إذا لم يدر كم صلى ، آداب عامة - الخطأ والنسيان ، إيمان - أعمال الجن والشياطين ، إيمان - الجن والشياطين']
    url=['https://dorar.net/hadith/sharh/215393', 'https://dorar.net/hadith/sharh/225405', 'https://dorar.net/hadith/sharh/225405']
    page=['1/360', '5/21', '1/360']
    sharh[:150]: فرض الله تعالى الصلاة، وبين النبي صلى الله عليه وسلم صفتها،

In [36]:
# Demonstration: per-hadith aligned filtering
# If a chunk has 10 hadiths and only 3 are from البخاري, only those 3 are returned.
query = "ايه هو سجود السهو؟"
results_filtered = retrieve_filtered(query, mohadth=["البخاري","مسلم"])
print(f"Filtered results: {len(results_filtered)}")
for r in results_filtered[:1]:
    print(f"  hadiths ({len(r.hadiths)}): rawy={r.list_meta.get('rawy')}")
    for j, h in enumerate(r.hadiths):
        print(f"    [{j}] rawy={r.list_meta['rawy'][j] if j < len(r.list_meta.get('rawy', [])) else '?'!r} | {h}")

Filtered results: 8
  hadiths (13): rawy=['أبو هريرة', 'أبو هريرة', 'أبو هريرة', 'أبو هريرة', 'أبو هريرة', 'أبو هريرة', 'أبو هريرة', 'أبو هريرة', 'أبو هريرة', 'أبو هريرة', 'أبو هريرة', 'أبو بكر بن سليمان بن أبي حثمة', 'أبو هريرة']
    [0] rawy='أبو هريرة' | - صَلَّى بنا رَسولُ اللهِ صلَّى اللهُ عليه وسلَّم إحدى صَلاتَيِ العَشيِّ -قال ابنُ سيرينَ: سَمَّاها أبو هُرَيرةَ ولَكِن نَسيتُ أنا- قال: فصَلَّى بنا رَكعَتَينِ، ثُمَّ سَلَّمَ، فقامَ إلى خَشَبةٍ مَعروضةٍ في المَسجِدِ، فاتَّكَأ عليها كَأنَّه غَضبانُ، ووضَعَ يَدَه اليُمنى على اليُسرى، وشَبَّكَ بينَ أصابِعِه، ووضَعَ خَدَّه الأيمَنَ على ظَهرِ كَفِّه اليُسرى، وخَرَجَتِ السَّرَعانُ مِن أبوابِ المَسجِدِ، فقالوا: قَصُرَتِ الصَّلاةُ؟ وفي القَومِ أبو بَكرٍ وعُمَرُ، فهابا أن يُكَلِّماه، وفي القَومِ رَجُلٌ في يَدَيه طولٌ، يُقالُ له: ذو اليَدَينِ، قال: يا رَسولَ اللهِ، أنَسيتَ أم قَصُرَتِ الصَّلاةُ؟ قال: لَم أنسَ ولَم تُقصَرْ، فقال: أكما يقولُ ذو اليَدَينِ؟ فقالوا: نَعَم، فتَقدَّمَ فصَلَّى ما تَرَكَ، ثُمَّ سَلَّمَ، ثُمَّ كَبَّرَ وسَجَدَ مِثلَ سُج

In [ ]:
# Example usage:
query = "ايه هو سجود السهو؟"
results_filtered = retrieve_filtered(query, mohadth=["البخاري","مسلم"])
trimmed = top_w_hadiths(results, w=3)  # at most 3 hadiths per chunk
for r in trimmed[:4]:
    # print(f"  hadiths ({len(r.hadiths)}): rawy={r.list_meta.get('rawy')}")
    # for j, h in enumerate(r.hadiths):
        # print(f"    [{j}] rawy={r.list_meta['rawy'][j] if j < len(r.list_meta.get('rawy', [])) else '?'!r} | {h}")
    # print("=" *150)
    print(r.hadiths)
    print("="*80)

# Evaluation

## Customization cheat sheet

| Goal | Change in `CONFIG` |
|------|-------------------|
| Use full dataset | `"max_rows": None` |
| Rebuild vector DB | `"reset_on_build": True` (run vector-store cell once) |
| More context per answer | Increase `retriever.k` or `chunk_size` |
| Diverse retrieval | `"search_type": "mmr"` |
| Local LLM | `"llm": {"provider": "ollama", ...}` + run Ollama |
| OpenAI embeddings | `"embeddings": {"provider": "openai", ...}` |
| Different fields in chunks | Edit `data.text_columns` / `metadata_columns` |
| Swap only the prompt | Edit `prompt.system_role` or the `RAG_PROMPT` cell |

In [37]:
import pandas as pd
eval_df = pd.read_csv("/kaggle/input/datasets/aliabdelmenam/gg-data/HAQA.csv")

In [38]:
eval_df.head(1)

,Record_Id,Question_Id,Question_Text,Quetion_Type,Question_Start_Word,Answer_ID,Full_Answer,Expert_Commentary,Hadith_Full_Answer,Hadith_Matn,Answer-Instances,Source_Name,Source_Link,Credibility,Question_ID_in_the_Orignal_Dataset
0,1,1,كيف نعبد الله؟,D,كيف,1.0,كَمَا أمرنا الله ورسوله مَعَ الإخلاص ( وَمَا أ...,كَمَا أمرنا الله ورسوله مَعَ الإخلاص,( مَن عمِلَ عمَلاً لَيْسَ عَلَيهِ أمْرُنا فَهُ...,مَن عمِلَ عمَلاً لَيْسَ عَلَيهِ أمْرُنا فَهُو...,مَن عمِلَ عمَلاً لَيْسَ عَلَيهِ أمْرُنا فَهُو...,كتاب عقيدة كل مسلم في سؤال و جواب لمحمد بن جمي...,https://www.noor-book.com/%D9%83%D8%AA%D8%A7%D...,yes,2


In [39]:
eval_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1597 entries, 0 to 1596
Data columns (total 15 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Record_Id                           1597 non-null   int64  
 1   Question_Id                         1597 non-null   int64  
 2   Question_Text                       1597 non-null   object 
 3   Quetion_Type                        1597 non-null   object 
 4   Question_Start_Word                 1597 non-null   object 
 5   Answer_ID                           1597 non-null   float64
 6   Full_Answer                         1597 non-null   object 
 7   Expert_Commentary                   1579 non-null   object 
 8   Hadith_Full_Answer                  1597 non-null   object 
 9   Hadith_Matn                         1597 non-null   object 
 10  Answer-Instances                    1597 non-null   object 
 11  Source_Name                         1597 no

In [40]:
eval_df[eval_df['Question_Id'] == 445]

,Record_Id,Question_Id,Question_Text,Quetion_Type,Question_Start_Word,Answer_ID,Full_Answer,Expert_Commentary,Hadith_Full_Answer,Hadith_Matn,Answer-Instances,Source_Name,Source_Link,Credibility,Question_ID_in_the_Orignal_Dataset
512,513,445,الاستجمار بكم يكون من الحجارة ؟,D,بكم,1.0,إذا استجمر فليوتر، ولا يكون بأقل من ثلاث يعني:...,إذا استجمر فليوتر، ولا يكون بأقل من ثلاث يعني:...,حديث أَبِي هُرَيْرَةَ رضي الله عنه، أَنَّ رَسُ...,أَنَّ رَسُولَ الله ﷺ قَالَ: مَنِ اسْتَجْمَرَ ف...,فَليُوتِرْ,الاستدلال على كنز الأطفال للدكتور فيصل بن مسفر...,https://alilmia.com/sub_book91_5410.html,yes,٧٦٥
513,514,445,الاستجمار بكم يكون من الحجارة ؟,D,بكم,2.0,إذا استجمر فليوتر، ولا يكون بأقل من ثلاث يعني:...,إذا استجمر فليوتر، ولا يكون بأقل من ثلاث يعني:...,عَنْ سَلمَانَ رضي الله عنه، قَالَ: «نَهَانَا ...,قَالَ: «نَهَانَا رَسُولُ الله ﷺ أَنْ نَسْتَنْج...,نَهَانَا رَسُولُ الله ﷺ أَنْ نَسْتَنْجِيَ بِأَ...,الاستدلال على كنز الأطفال للدكتور فيصل بن مسفر...,https://alilmia.com/sub_book91_5410.html,yes,٧٦٥


## Evaluation config

In [41]:
from datetime import datetime, timezone
from pathlib import Path

EVAL_CONFIG = {
    "haqa_csv": "/kaggle/input/datasets/aliabdelmenam/gg-data/HAQA.csv",
    "max_samples": None,
    "random_seed": 42,
    "output_dir": PROJECT_ROOT / "eval_runs",
    "run_name": None,
    # Hadith-in-retrieval
    "retrieval": {
        "enabled": True,
        "min_substring_len": 12,
        "min_token_overlap": 0.45,
        "check_metadata_hadith_1": True,
    },
    # BLEU / ROUGE
    "bleu_rouge": {
        "enabled": True,
        # Strategy: compare ground-truth hadith against each retrieved hadith
        # individually and take the BEST score — correct when retrieval returns
        # many hadiths per chunk and the true answer is just one of them.
        "strategy": "best_match",   # "best_match" | "concatenate"
    },
    # RAGAS
    "ragas": {
        "enabled": True,
        "metrics": [
            "faithfulness",
            "answer_relevancy",
            "context_precision",
            "context_recall",
        ],
        "ground_truth_column": "Expert_Commentary",
        "max_contexts_for_ragas": None,
    },
}

def _eval_run_dir() -> Path:
    name = EVAL_CONFIG["run_name"] or datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    run_dir = EVAL_CONFIG["output_dir"] / name
    run_dir.mkdir(parents=True, exist_ok=True)
    return run_dir

EVAL_RUN_DIR = _eval_run_dir()
print(f"Eval outputs -> {EVAL_RUN_DIR}")

Eval outputs -> /kaggle/working/eval_runs/20260625_224905


In [42]:
# The error occurred because 'retrieve' was not defined in the current scope.
# Ensure cell 23371392 (which defines retrieve) has been executed.

try:
    koks = retrieve_filtered(query="ما حكم تربيةالدقن")
    print(f"Successfully retrieved {len(koks)} results.")
except NameError:
    print("Error: The 'retrieve' function is not defined. Please run the cell containing the RAG chain definition first.")

Successfully retrieved 8 results.


In [43]:
# FIX: RetrievedResult uses .hadiths and .list_meta instead of .metadata
if koks and len(koks) > 0:
    print(f"Printing hadiths for the first result (Total: {len(koks[0].hadiths)}):")
    for hadith in koks[0].hadiths[::-1]:
        print(f"- {hadith}")
else:
    print("No results found in 'koks'.")

Printing hadiths for the first result (Total: 4):
- أنَّ ابنَ عباسٍ سُئِلَ عمَّن ذَبَحَ دجاجةً فطيَّرَ رأسَها فقال ذكاةٌ وحيَّةٌ
- - أنَّ ابنَ عبَّاسٍ سُئلَ عمَّن ذبحَ دجاجةً فطَيَّرَ رأسَها فقالَ ذَكاةٌ وَحِيَّةٌ
- - عن ابنِ عبَّاسٍ سُئِلَ عن ذَبحِ دَجاجَةٍ طُيِّرَ رأسُها، فقال: ذَكاةٌ وَحِيَّةٌ.
- - أنَّ ابنَ عباسٍ سُئِلَ عمَّن ذَبَحَ دجاجةً فطيَّرَ رأسَها فقال ذكاةٌ وحيَّةٌ


In [44]:
all_hadiths = []
for doc in koks:
    # Use the .hadiths attribute instead of .metadata['hadith']
    hadiths = doc.hadiths
    print(f"Found {len(hadiths)} hadiths in this doc")
    all_hadiths.extend(hadiths)

print(f"\nTotal hadiths across {len(koks)} docs: {len(all_hadiths)}")

Found 4 hadiths in this doc
Found 3 hadiths in this doc
Found 9 hadiths in this doc
Found 3 hadiths in this doc
Found 12 hadiths in this doc
Found 7 hadiths in this doc
Found 3 hadiths in this doc
Found 3 hadiths in this doc

Total hadiths across 8 docs: 44


In [45]:
import re
from dataclasses import dataclass




def token_set(text: str) -> set[str]:
    return {t for t in normalize_arabic(text).split() if len(t) > 1}


def token_overlap_ratio(a: str, b: str) -> float:
    ta, tb = token_set(a), token_set(b)
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / len(ta)


def sequence_matcher(a: str, b: str) -> float:
    from difflib import SequenceMatcher
    return SequenceMatcher(None, normalize_arabic(a), normalize_arabic(b)).ratio()

In [46]:
# --- eval_retrieval_hadith.py ---
@dataclass
class HadithHitResult:
    hit: bool
    method: str | None          # substring | token_overlap | none
    best_overlap: float
    matched_doc_index: str | None


def hadith_in_retrieved_docs(
    hadith_matn: str,
    all_hadith: list[str], * , 
    min_substring_len: int | None = None, min_token_overlap: float | None = None) -> HadithHitResult:


    min_substring_len = min_substring_len or EVAL_CONFIG["retrieval"]["min_substring_len"]
    min_token_overlap = min_token_overlap or EVAL_CONFIG["retrieval"]["min_token_overlap"]

    target = normalize_arabic(hadith_matn)
    if len(target) < 3:
        return HadithHitResult(False, "none", 0.0, None)

    best_overlap = 0.0
    best_hadith = ""
    for i, hadith in enumerate(all_hadith):
        blob_norm = normalize_arabic(hadith)
        if len(target) >= min_substring_len and target in blob_norm:
            return HadithHitResult(True, "substring", 1.0, blob_norm)
        # overlap = token_overlap_ratio(hadith_matn, doc_retrieval_blob(doc))
        overlap = sequence_matcher(target, blob_norm)
        if overlap > best_overlap:
            best_overlap, best_hadith = overlap, blob_norm

    if best_overlap >= min_token_overlap:
        return HadithHitResult(True, "token_overlap", best_overlap, best_hadith)
    return HadithHitResult(False, "none", best_overlap, best_hadith)


def load_haqa_eval_frame() -> pd.DataFrame:
    path = EVAL_CONFIG["haqa_csv"]
    df = pd.read_csv(path)
    df = df.dropna(subset=["Question_Text", "Hadith_Matn"])
    max_n = EVAL_CONFIG["max_samples"]
    if max_n:
        df = df.sample(n=min(max_n, len(df)), random_state=EVAL_CONFIG["random_seed"])
    return df.reset_index(drop=True)

In [47]:
def run_haqa_retrieval_eval(haqa_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in tqdm(haqa_df.iterrows(), total=len(haqa_df)):
        question = row["Question_Text"]
        hadith = row["Hadith_Matn"]

        # Retrieve filtered results
        # Using retrieve_filtered returns RetrievedResult objects
        docs = retrieve_filtered(query=normalize_arabic(question), mohadth=["البخاري","مسلم"])
        # docs = top_w_hadiths(docs, w=3)
        all_hadiths = []
        import ast
        for doc in docs:
            # Directly access the .hadiths attribute from the RetrievedResult object
            hadithss = doc.hadiths
            for ha in hadithss:
                all_hadiths.append(ha)
        hit = hadith_in_retrieved_docs(hadith, all_hadiths)
        rows.append({
            "Record_Id": row.get("Record_Id"),
            "Question_Id": row.get("Question_Id"),
            "Question_Text": question,
            "Hadith_Matn": hadith,
            "hadith_hit": hit.hit,
            "hit_method": hit.method,
            "best_token_overlap": round(hit.best_overlap, 4),
            # "matched_rank": (hit.matched_doc_index + 1) if hit.matched_doc_index is not None else None,
            # "retrieved_page_ids": [d.metadata.get("page_id") for d in docs],
            "n_retrieved": len(docs),
            "hadith":hit.matched_doc_index
        })
    return pd.DataFrame(rows)

if EVAL_CONFIG["retrieval"]["enabled"]:
    haqa_eval_df = load_haqa_eval_frame()
    retrieval_results_df = run_haqa_retrieval_eval(haqa_eval_df)

    hit_rate = retrieval_results_df["hadith_hit"].mean()
    print(f"Hadith-in-retrieval hit rate: {hit_rate:.1%} ({retrieval_results_df['hadith_hit'].sum()}/{len(retrieval_results_df)})")
    print(retrieval_results_df["hit_method"].value_counts(dropna=False))

    out_path = EVAL_RUN_DIR / "haqa_retrieval_hits.csv"
    retrieval_results_df.to_csv(out_path, index=False, encoding="utf-8-sig")

    import json
    summary_path = EVAL_RUN_DIR / "run_summary.json"
    summary_path.write_text(
        json.dumps(
            {
                "run_dir": str(EVAL_RUN_DIR),
                "n_samples": len(retrieval_results_df),
                "config_snapshot": CONFIG,
                "eval_config": {k: v for k, v in EVAL_CONFIG.items() if k != "haqa_csv"},
                "hadith_hit_rate": float(hit_rate),
                "hit_method_counts": retrieval_results_df["hit_method"].value_counts().to_dict(),
            },
            ensure_ascii=False,
            indent=2,
            default=str,
        ),
        encoding="utf-8",
    )
    print(f"Saved -> {out_path}, {summary_path}")
    retrieval_results_df.head()
else:
    print("Retrieval eval disabled in EVAL_CONFIG")

100%|██████████| 1597/1597 [01:59<00:00, 13.41it/s]

Hadith-in-retrieval hit rate: 42.1% (672/1597)
hit_method
none             925
token_overlap    662
substring         10
Name: count, dtype: int64
Saved -> /kaggle/working/eval_runs/20260625_224905/haqa_retrieval_hits.csv, /kaggle/working/eval_runs/20260625_224905/run_summary.json


In [48]:
if 'retrieval_results_df' in globals():
    null_hits = retrieval_results_df[retrieval_results_df['hadith_hit'].isna()]
    print(f"Number of null 'hadith_hit' values: {len(null_hits)}")
    if not null_hits.empty:
        display(null_hits)
    else:
        print("No Null values found in 'hadith_hit' column.")
else:
    print("Error: 'retrieval_results_df' not found. Please run the evaluation cell first.")

Number of null 'hadith_hit' values: 0
No Null values found in 'hadith_hit' column.


## BLEU & ROUGE Evaluation (Fix #2)

**The challenge:** the ground-truth has one hadith, but each retrieved chunk may
contain many hadiths (since we grouped by `sharh`).

**Strategy — `best_match`:**  
For each evaluation question, embed the ground-truth hadith and compare it
against every retrieved hadith individually. Report the **maximum** BLEU and
ROUGE score across all comparisons.  This is the correct choice because:
- Retrieval success means the right hadith is *somewhere* in the results.
- Averaging over all retrieved hadiths would dilute the score by unrelated hadiths.
- Taking the best match measures whether the pipeline found the correct hadith at all.

Macro-averages across questions are also reported.

In [49]:
!pip install nltk rouge-score --quiet

In [50]:
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

def rouge1_recall(reference: str, hypothesis: str) -> float:
    """How much of the reference is covered by the hypothesis."""
    ref_tokens = normalize_arabic(reference).split()
    hyp_tokens = set(normalize_arabic(hypothesis).split())
    if not ref_tokens:
        return 0.0
    matched = sum(1 for t in ref_tokens if t in hyp_tokens)
    return matched / len(ref_tokens)

def bleu1_precision(reference: str, hypothesis: str) -> float:
    """How much of the hypothesis actually appears in the reference."""
    ref_tokens = set(normalize_arabic(reference).split())
    hyp_tokens = normalize_arabic(hypothesis).split()
    if not hyp_tokens:
        return 0.0
    matched = sum(1 for t in hyp_tokens if t in ref_tokens)
    return matched / len(hyp_tokens)

def compute_best_match_scores(ground_truth_hadith: str, retrieved_hadiths: list[str]) -> dict[str, float]:
    if not retrieved_hadiths:
        return {"bleu": 0.0, "rouge1": 0.0}
    best = {"bleu": 0.0, "rouge1": 0.0}
    for hyp in retrieved_hadiths:
        # Using precision as a proxy for BLEU-1 and recall for ROUGE-1 per your request
        prec = bleu1_precision(ground_truth_hadith, hyp)
        rec = rouge1_recall(ground_truth_hadith, hyp)

        if prec > best["bleu"]:
            best["bleu"] = prec
        if rec > best["rouge1"]:
            best["rouge1"] = rec

    return best

print("Custom recall/precision metrics applied.")

Custom recall/precision metrics applied.


In [51]:
def run_bleu_rouge_eval(haqa_df: pd.DataFrame) -> pd.DataFrame:
    """
    Runs evaluation using custom precision (bleu) and recall (rouge1) metrics.
    """
    strategy = EVAL_CONFIG["bleu_rouge"]["strategy"]
    rows = []

    for _, row in tqdm(haqa_df.iterrows(), total=len(haqa_df), desc="Precision/Recall Eval"):
        question     = row["Question_Text"]
        ground_truth = row["Hadith_Matn"]

        # Retrieve filtered results with metadata constraint
        docs = retrieve_filtered(normalize_arabic(question), mohadth=["البخاري","مسلم"])
        # docs = top_w_hadiths(docs, w=3)
        retrieved_hadiths: list[str] = []
        for doc in docs:
            h = doc.hadiths
            retrieved_hadiths.extend(normalize_arabic(str(x)) for x in h)

        gt_norm = normalize_arabic(ground_truth)

        if strategy == "best_match":
            best_scores = compute_best_match_scores(gt_norm, retrieved_hadiths)
        else:
            # Defaulting to best_match for custom metrics as requested
            best_scores = compute_best_match_scores(gt_norm, retrieved_hadiths)

        rows.append({
            "Record_Id":           row.get("Record_Id"),
            "Question_Id":         row.get("Question_Id"),
            "Question_Text":       question,
            "Hadith_Matn":         ground_truth,
            "n_retrieved_hadiths": len(retrieved_hadiths),
            "precision_bleu":      round(best_scores["bleu"],   4),
            "recall_rouge1":       round(best_scores["rouge1"], 4),
        })

    return pd.DataFrame(rows)

def print_custom_summary(df: pd.DataFrame) -> None:
    print("=" * 55)
    print("Custom Metrics Summary (Best Match Strategy)")
    print(f"Precision (proxy for BLEU-1): {df['precision_bleu'].mean():.4f}")
    print(f"Recall (proxy for ROUGE-1):    {df['recall_rouge1'].mean():.4f}")
    print("=" * 55)

if EVAL_CONFIG["bleu_rouge"]["enabled"]:
    if "haqa_eval_df" not in globals():
        haqa_eval_df = load_haqa_eval_frame()

    bleu_rouge_df = run_bleu_rouge_eval(haqa_eval_df)
    print_custom_summary(bleu_rouge_df)

    bleu_rouge_path = EVAL_RUN_DIR / "haqa_custom_metrics.csv"
    bleu_rouge_df.to_csv(bleu_rouge_path, index=False, encoding="utf-8-sig")
    display(bleu_rouge_df.head())

Precision/Recall Eval: 100%|██████████| 1597/1597 [01:44<00:00, 15.27it/s]

Custom Metrics Summary (Best Match Strategy)
Precision (proxy for BLEU-1): 0.4366
Recall (proxy for ROUGE-1):    0.5106


,Record_Id,Question_Id,Question_Text,Hadith_Matn,n_retrieved_hadiths,precision_bleu,recall_rouge1
0,1,1,كيف نعبد الله؟,مَن عمِلَ عمَلاً لَيْسَ عَلَيهِ أمْرُنا فَهُو...,28,0.0690,0.6250
1,2,2,هل نعبد الله خوفا وطمعا؟,أسألُ اللهَ الجنّة وأعوذ بِهِ مِن النّار,35,0.3333,0.7143
2,3,3,ما هو الإحسان في العبادة؟,الإحسانُ أَنْ تعبُدَ اللهَ كأنّك تراه فإن لَم...,37,0.3333,0.9167
3,4,4,ما معنى لا إله إلا الله؟,من قَالَ لآ إله إِلاَّ الله وكَفَرَ بِمَا يُع...,44,0.4545,0.6000
4,5,5,ما هو التوحيد في صفات الله؟,ينزِلُ ربُّنا تبارك وتعالى فِي كلّ ليلةٍ إِلَى...,21,0.0430,0.2000


# W&B

In [ ]:
!pip install wandb
!pip install --upgrade wandb

In [ ]:
import os
import wandb
from kaggle_secrets import UserSecretsClient

# Fetch the key
user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("W&B_API")

# Set it globally in the environment
os.environ["WANDB_API_KEY"] = wandb_key
os.environ["WANDB_SILENT"] = "true"

In [ ]:
!wandb login

In [ ]:
import wandb

run = wandb.init(
    entity="RAG-Hadith",   # your team name
    project="temp project",       # shared project — same for everyone
    name="First Try (Normal chunking)",  # descriptive run name
    config={
        # Retrieval / Data parameters
        "embedding_model":  cfg("embeddings", "model_name"),
        "chunk_size":       "None",
        "chunk_overlap":    "None",
        "k":                cfg("retriever", "k"),
        
        "min_token_overlap": ".5",
        "description": "use normal chunking and hadith is from بخاري او مسلم only, but this time i have only chosed the top 3 longest hadith not all بخاري اي مسلم hadiths"
    }
)

In [ ]:
wandb.log({
    "Hit Rate": 33.2,
    "BELU": 0.4357,
    "ROUGE":0.4871
})

## RAGAS

In [ ]:
!pip install "ragas==0.2.15" "langchain-community==0.2.19" --quiet

In [ ]:
import json

from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    answer_relevancy,
    context_precision,
    context_recall,
    faithfulness,
)

RAGAS_METRIC_MAP = {
    "faithfulness": faithfulness,
    "answer_relevancy": answer_relevancy,
    "context_precision": context_precision,
    "context_recall": context_recall,
}


def _ground_truth_from_row(row: pd.Series) -> str:
    col = EVAL_CONFIG["ragas"]["ground_truth_column"]
    if col in row.index and pd.notna(row[col]) and str(row[col]).strip():
        return str(row[col])
    if pd.notna(row.get("Full_Answer")):
        return str(row["Full_Answer"])
    return str(row.get("Expert_Commentary", ""))


def build_ragas_dataset(haqa_df: pd.DataFrame) -> Dataset:
    rows = []
    max_ctx = EVAL_CONFIG["ragas"]["max_contexts_for_ragas"]
    for _, row in haqa_df.iterrows():
        question = row["Question_Text"]
        docs = retrieve(question)
        contexts = [d.page_content for d in docs]
        if max_ctx:
            contexts = contexts[:max_ctx]
        answer = rag_chain.invoke(question)
        rows.append({
            "question": question,
            "answer": answer,
            "contexts": contexts,
            "ground_truth": _ground_truth_from_row(row),
            "hadith_matn": row["Hadith_Matn"],
        })
    return Dataset.from_list(rows)


def run_ragas_eval(haqa_df: pd.DataFrame):
    metric_names = EVAL_CONFIG["ragas"]["metrics"]
    metrics = [RAGAS_METRIC_MAP[m] for m in metric_names]

    dataset = build_ragas_dataset(haqa_df)
    result = evaluate(
        dataset=dataset,
        metrics=metrics,
        llm=llm,
        embeddings=embeddings,
    )
    return result


if EVAL_CONFIG["ragas"]["enabled"]:
    if "haqa_eval_df" not in globals():
        haqa_eval_df = load_haqa_eval_frame()

    ragas_result = run_ragas_eval(haqa_eval_df)
    ragas_per_question_df = ragas_result.to_pandas()

    per_row_path = EVAL_RUN_DIR / "ragas_per_question.csv"
    ragas_per_question_df.to_csv(per_row_path, index=False, encoding="utf-8-sig")

    metric_cols = [c for c in ragas_per_question_df.columns if c in EVAL_CONFIG["ragas"]["metrics"]]
    ragas_mean = ragas_per_question_df[metric_cols].mean(numeric_only=True).to_dict()

    # Combine all eval results into one summary
    summary: dict = {
        "run_dir": str(EVAL_RUN_DIR),
        "n_samples": len(haqa_eval_df),
        "config_snapshot": CONFIG,
        "eval_config": {k: v for k, v in EVAL_CONFIG.items() if k != "haqa_csv"},
        "ragas_mean": ragas_mean,
    }
    if "retrieval_results_df" in globals():
        summary["hadith_hit_rate"] = float(retrieval_results_df["hadith_hit"].mean())
    if "bleu_rouge_df" in globals():
        summary["bleu_rouge_mean"] = {
            col: round(float(bleu_rouge_df[col].mean()), 4)
            for col in bleu_rouge_df.columns
            if col.startswith(("best_", "concat_"))
        }

    summary_path = EVAL_RUN_DIR / "run_summary.json"
    summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

    print("RAGAS (mean over samples):")
    for k, v in ragas_mean.items():
        print(f"  {k}: {v:.4f}")
    print(f"Saved -> {per_row_path}, {summary_path}")
    ragas_per_question_df.head()
else:
    print("RAGAS eval disabled in EVAL_CONFIG")

In [ ]:
import json


def load_eval_summaries(runs_root: Path | None = None) -> pd.DataFrame:
    runs_root = runs_root or EVAL_CONFIG["output_dir"]
    rows = []
    for p in sorted(runs_root.glob("*/run_summary.json")):
        data = json.loads(p.read_text(encoding="utf-8"))
        row = {"run": p.parent.name}
        row.update(data.get("ragas_mean", {}))
        row["hadith_hit_rate"] = data.get("hadith_hit_rate")
        row.update(data.get("bleu_rouge_mean", {}))
        rows.append(row)
    return pd.DataFrame(rows)


# Example after 2+ trials:
# compare_df = load_eval_summaries()
# compare_df.sort_values("hadith_hit_rate", ascending=False)